# GESAMTSTATIK — Cesium → IFC → GMSH → JaxFEM

**Pipeline:**
```
IFC-Datei (Server / lokal)
   └─ IfcOpenShell  → Bauteil-Geometrien (BRep, OCC-Kernel)
       └─ GMSH      → Tet4-Volumennetz
           └─ meshio → numpy arrays (points, cells)
               └─ JaxFEM → Spannungs-/Verformungsberechnung
```

Unterstützte IFC-Typen: `IfcBeam`, `IfcColumn`, `IfcSlab`, `IfcWall`

**Konfiguration:** Nur Zelle 2 (CONFIG) anpassen, dann alle Zellen ausführen.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as onp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os, sys, tempfile, json, pathlib

import ifcopenshell
import ifcopenshell.geom
import gmsh
import meshio

from jax_fem.generate_mesh import Mesh, get_meshio_cell_type
from jax_fem.problem import Problem
from jax_fem.solver import solver
from jax_fem import logger_setup

logger_setup.setup_logger('jax_fem')
print('JAX      :', jax.__version__)
print('meshio   :', meshio.__version__)
print('gmsh     :', gmsh.__version__)
print('ifcOS    :', ifcopenshell.__version__)

## 0 — Konfiguration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  HIER ANPASSEN                                               ║
# ╚══════════════════════════════════════════════════════════════╝

# Pfad zur IFC-Datei (lokal oder nach Download vom Server)
IFC_PATH = "../data/gesamtstatik.ifc"

# Falls kein IFC vorhanden: synthetisches Demo-Modell erzeugen?
USE_DEMO_MODEL = True

# IFC-Entitäten, die in die Berechnung eingehen
IFC_FILTER = ['IfcBeam', 'IfcColumn', 'IfcSlab', 'IfcWall']

# GMSH-Netzfeinheit [m]
MESH_SIZE = 0.15

# Materialparameter (Beton C25/30, kN + m)
E_DEFAULT  = 31_000.0   # kN/m²
NU_DEFAULT = 0.2

# Gleichmäßige Flächenlast auf Oberkante [kN/m²]
Q_LOAD = 10.0

# Ausgabeordner
DATA_DIR = pathlib.Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'vtk').mkdir(exist_ok=True)
(DATA_DIR / 'msh').mkdir(exist_ok=True)

print('Konfiguration geladen.')
print(f'  IFC_PATH   : {IFC_PATH}')
print(f'  USE_DEMO   : {USE_DEMO_MODEL}')
print(f'  MESH_SIZE  : {MESH_SIZE} m')
print(f'  E / nu     : {E_DEFAULT} kN/m²  /  {NU_DEFAULT}')

## 1 — IFC laden & Geometrie extrahieren

### 1a — Demo-Modell (synthetisch, kein IFC nötig)

Erzeugt ein minimales IFC-Modell mit:
- 1 × `IfcBeam`   (Einfeldträger L=6 m, b=0.3 m, h=0.5 m)
- 2 × `IfcColumn` (Stützen h=3 m, 0.3×0.3 m)

Das Demo-IFC wird als `data/gesamtstatik_demo.ifc` gespeichert und dann
exakt wie eine echte IFC-Datei weiterverarbeitet.

In [ ]:
def create_demo_ifc(path: str) -> str:
    """Minimales IFC4-Modell: 1 Balken + 2 Stützen.
    Koordinaten in MILLIMETERN (IFC-Standard), damit IfcOpenShell
    geom.create_shape reale Abmessungen liefert."""
    import ifcopenshell.api

    model   = ifcopenshell.api.run('project.create_file', version='IFC4')
    project = ifcopenshell.api.run('root.create_entity', model, ifc_class='IfcProject', name='Demo')
    # IFC-Standard-Einheit = mm
    ifcopenshell.api.run('unit.assign_unit', model, length={'is_metric': True, 'raw': 'MILLIMETRES'})

    ctx  = ifcopenshell.api.run('context.add_context', model, context_type='Model')
    body = ifcopenshell.api.run('context.add_context', model,
                                 context_type='Model', context_identifier='Body',
                                 target_view='MODEL_VIEW', parent=ctx)

    site     = ifcopenshell.api.run('root.create_entity', model, ifc_class='IfcSite',     name='Site')
    building = ifcopenshell.api.run('root.create_entity', model, ifc_class='IfcBuilding', name='Building')
    storey   = ifcopenshell.api.run('root.create_entity', model, ifc_class='IfcBuildingStorey', name='EG')
    ifcopenshell.api.run('aggregate.assign_object', model, products=[site],     relating_object=project)
    ifcopenshell.api.run('aggregate.assign_object', model, products=[building], relating_object=site)
    ifcopenshell.api.run('aggregate.assign_object', model, products=[storey],   relating_object=building)

    def _box_verts_faces(x0, y0, z0, dx, dy, dz):
        v = [(x0,y0,z0),(x0+dx,y0,z0),(x0+dx,y0+dy,z0),(x0,y0+dy,z0),
             (x0,y0,z0+dz),(x0+dx,y0,z0+dz),(x0+dx,y0+dy,z0+dz),(x0,y0+dy,z0+dz)]
        f = [(0,1,2),(0,2,3),(4,6,5),(4,7,6),(0,1,5),(0,5,4),
             (2,3,7),(2,7,6),(0,3,7),(0,7,4),(1,2,6),(1,6,5)]
        return v, f

    def add_elem(ifc_class, name, x0, y0, z0, dx, dy, dz):
        elem = ifcopenshell.api.run('root.create_entity', model, ifc_class=ifc_class, name=name)
        verts, faces = _box_verts_faces(x0, y0, z0, dx, dy, dz)
        rep = ifcopenshell.api.run('geometry.add_mesh_representation', model,
                                    context=body, vertices=[verts], faces=[faces])
        ifcopenshell.api.run('geometry.assign_representation', model, product=elem, representation=rep)
        ifcopenshell.api.run('spatial.assign_container', model,
                              relating_structure=storey, products=[elem])

    # Alle Koordinaten in mm: Balken L=6000, b=300, h=500; Stützen h=3000, 300×300
    add_elem('IfcBeam',   'Balken_B1',     0,   0, 3000, 6000, 300,  500)
    add_elem('IfcColumn', 'Stuetze_C1',    0,   0,    0,  300, 300, 3000)
    add_elem('IfcColumn', 'Stuetze_C2', 5700,   0,    0,  300, 300, 3000)

    model.write(path)
    print(f'Demo-IFC gespeichert: {path}  (Einheit: mm)')
    return path


def _detect_unit_scale(ifc_model):
    """Liest die IFC-Längeneinheit und gibt den Faktor → m zurück."""
    try:
        for unit_assignment in ifc_model.by_type('IfcUnitAssignment'):
            for unit in unit_assignment.Units:
                if not hasattr(unit, 'UnitType') or unit.UnitType != 'LENGTHUNIT':
                    continue
                prefix = getattr(unit, 'Prefix', None)
                name   = getattr(unit, 'Name', '')
                if prefix == 'MILLI' or 'MILLI' in str(name).upper():
                    return 0.001
                if prefix is None and str(name).upper() in ('METRE', 'METER'):
                    return 1.0
    except Exception:
        pass
    return 0.001  # IFC-Fallback = mm


# ── IFC bestimmen ─────────────────────────────────────────────────────────────
if USE_DEMO_MODEL or not os.path.exists(IFC_PATH):
    if not USE_DEMO_MODEL:
        print(f'WARNUNG: {IFC_PATH} nicht gefunden → Demo-Modell wird verwendet')
    ifc_path_actual = str(DATA_DIR / 'gesamtstatik_demo.ifc')
    create_demo_ifc(ifc_path_actual)
else:
    ifc_path_actual = IFC_PATH

ifc_model = ifcopenshell.open(ifc_path_actual)
IFC_UNIT_TO_M = _detect_unit_scale(ifc_model)
print(f'IFC geladen: {ifc_path_actual}')
print(f'Längeneinheit-Faktor → m: {IFC_UNIT_TO_M}')
for et in IFC_FILTER:
    count = len(ifc_model.by_type(et))
    print(f'  {et:20s}: {count} Objekte')

### 1b — Geometrie-Extraktion via IfcOpenShell

IfcOpenShell parst die IFC-Geometrie über seinen internen OCC-Kernel.  
Für jedes Bauteil werden Vertices + Faces extrahiert und
einheitenbereinigt auf [m] umgerechnet.

In [ ]:
settings = ifcopenshell.geom.settings()
settings.set(settings.USE_WORLD_COORDS, True)

elements = []
for ifc_type in IFC_FILTER:
    elements.extend(ifc_model.by_type(ifc_type))

print(f'{len(elements)} Bauteile zur Vernetzung gefunden.')
print(f'Einheitenfaktor: {IFC_UNIT_TO_M}  (IFC-Koordinaten → m)')

geom_data = []

for elem in elements:
    try:
        shape = ifcopenshell.geom.create_shape(settings, elem)
    except Exception as e:
        print(f'  SKIP {elem.Name}: {e}')
        continue

    verts = onp.array(shape.geometry.verts).reshape(-1, 3) * IFC_UNIT_TO_M
    faces = onp.array(shape.geometry.faces).reshape(-1, 3)

    if len(verts) == 0:
        print(f'  SKIP {elem.Name}: keine Vertices')
        continue

    bbox_min = verts.min(axis=0)
    bbox_max = verts.max(axis=0)
    dims = bbox_max - bbox_min

    if dims.min() < 0.005:
        print(f'  WARN {elem.Name}: BBox sehr klein {dims.round(3)} m — Einheitenproblem?')

    geom_data.append({
        'name'  : elem.Name or elem.GlobalId,
        'type'  : elem.is_a(),
        'guid'  : elem.GlobalId,
        'verts' : verts,
        'faces' : faces,
        'bbox'  : (bbox_min, bbox_max),
        'dims'  : dims,
    })
    print(f'  {elem.is_a():12s}  {elem.Name or elem.GlobalId:20s}  '
          f'Knoten={len(verts):4d}  Dreiecke={len(faces):4d}  '
          f'BBox={dims[0]:.3f}×{dims[1]:.3f}×{dims[2]:.3f} m')

print(f'\n{len(geom_data)} Geometrien extrahiert.')

## 2 — Gesamt-Mesh via GMSH (Tet4)

Alle Bauteil-Oberflächen werden in eine GMSH-Session geladen.  
GMSH erzeugt ein gemeinsames Tet4-Volumennetz der Gesamtgeometrie.

In [ ]:
def build_gmsh_mesh(geom_list, mesh_size, out_msh):
    """Tet4-Netz aus IFC-Oberflächengeometrie via GMSH discrete modelling.

    Jedes Bauteil wird als eigene discrete surface + volume angelegt.
    Knoten-Tags werden bauteil-weise versetzt, damit sie sich nicht überschneiden.
    """
    if gmsh.isInitialized():
        gmsh.finalize()
    gmsh.initialize()
    gmsh.option.setNumber('General.Terminal', 0)
    gmsh.option.setNumber('Mesh.CharacteristicLengthMax', mesh_size)
    gmsh.option.setNumber('Mesh.CharacteristicLengthMin', mesh_size * 0.1)
    gmsh.option.setNumber('Mesh.Algorithm3D', 4)   # Frontal-Delaunay
    gmsh.model.add('gesamtstatik')

    node_offset = 0   # globaler Knoten-Tag-Offset pro Bauteil

    for gd in geom_list:
        verts = gd['verts']
        faces = gd['faces']
        n_v   = len(verts)
        n_f   = len(faces)

        if n_v < 4 or n_f < 4:
            print(f'  SKIP {gd["name"]}: zu wenig Geometrie ({n_v} Knoten, {n_f} Dreiecke)')
            continue

        # ── 1. Discrete surface anlegen ──────────────────────────────────────
        surf_tag = gmsh.model.addDiscreteEntity(2)

        # ── 2. Knoten eintragen (global eindeutige Tags via Offset) ──────────
        node_tags = [node_offset + i + 1 for i in range(n_v)]
        coords    = verts.flatten().tolist()
        params    = []   # keine parametrischen Koordinaten
        gmsh.model.mesh.addNodes(2, surf_tag, node_tags, coords, params)

        # ── 3. Dreieckselemente eintragen ────────────────────────────────────
        elem_tags  = [node_offset + i + 1 for i in range(n_f)]
        conn_flat  = (faces + node_offset + 1).flatten().tolist()   # 1-basiert
        gmsh.model.mesh.addElementsByType(surf_tag, 2, elem_tags, conn_flat)

        # ── 4. Oberfläche klassifizieren → geschlossenes Volumen ─────────────
        gmsh.model.mesh.classifySurfaces(onp.pi, True, True, onp.pi)
        gmsh.model.mesh.createGeometry()

        node_offset += n_v

    # ── 5. Alle Volumes finden und vernetzen ──────────────────────────────────
    vols = gmsh.model.getEntities(3)
    if not vols:
        gmsh.finalize()
        raise RuntimeError('GMSH: keine Volumes nach classifySurfaces — Geometrie nicht wasserdicht?')

    print(f'  GMSH: {len(vols)} Volume(s) gefunden, starte Tet4-Vernetzung …')
    gmsh.model.mesh.generate(3)
    gmsh.model.mesh.setOrder(1)
    try:
        gmsh.model.mesh.optimize('Netgen')
    except Exception:
        pass   # Netgen-Optimierung optional

    gmsh.write(str(out_msh))
    n_nodes = len(gmsh.model.mesh.getNodes()[0])
    n_tets  = len(gmsh.model.mesh.getElementsByType(4)[0])
    gmsh.finalize()
    print(f'  GMSH-Netz: {n_nodes} Knoten, {n_tets} Tet4-Elemente')
    return str(out_msh)


msh_path = DATA_DIR / 'msh' / 'gesamtstatik.msh'

print('Starte GMSH-Vernetzung …')
try:
    build_gmsh_mesh(geom_data, MESH_SIZE, msh_path)
    print(f'Netz gespeichert: {msh_path}')
    gmsh_ok = True
except Exception as e:
    print(f'GMSH-Fehler: {e}')
    print('→ Fallback: Box-Netz aus IFC-BBoxen (HEX8)')
    gmsh_ok = False

### 2b — Fallback: Box-Netz aus Bounding-Boxes

Falls GMSH die triangulierte IFC-Oberfläche nicht schließen kann  
(passiert bei fehlerhaften IFC-Geometrien), wird für jedes Bauteil  
ein reguläres Hex8-Netz aus der Bounding-Box erzeugt.

In [ ]:
from jax_fem.generate_mesh import box_mesh

def bbox_hex_mesh(geom_list, mesh_size):
    """Hex8-Netz pro Bauteil aus BBox, dann zusammengeführt."""
    all_pts   = []
    all_cells = []
    offset = 0

    for gd in geom_list:
        bmin, bmax = gd['bbox']
        Lx, Ly, Lz = (bmax - bmin)
        if min(Lx, Ly, Lz) < 1e-4:
            continue

        nx = max(2, int(Lx / mesh_size))
        ny = max(2, int(Ly / mesh_size))
        nz = max(2, int(Lz / mesh_size))

        m    = box_mesh(Nx=nx, Ny=ny, Nz=nz, domain_x=Lx, domain_y=Ly, domain_z=Lz)
        pts  = m.points + bmin
        # box_mesh liefert 'hexahedron' als Schlüssel (meshio-Standard)
        key  = 'hexahedron' if 'hexahedron' in m.cells_dict else list(m.cells_dict.keys())[0]
        cells = m.cells_dict[key] + offset

        all_pts.append(pts)
        all_cells.append(cells)
        offset += len(pts)
        print(f'  BBox-Netz {gd["name"]:20s}: {len(pts):6d} Knoten  {len(cells):6d} Elemente')

    pts_all   = onp.vstack(all_pts)
    cells_all = onp.vstack(all_cells)
    return pts_all, cells_all, 'HEX8'


if gmsh_ok:
    raw_mesh = meshio.read(str(msh_path))
    if 'tetra' in raw_mesh.cells_dict:
        points_m  = raw_mesh.points
        cells_m   = raw_mesh.cells_dict['tetra']
        ele_type  = 'TET4'
        print(f'Tet4-Netz: {len(points_m)} Knoten, {len(cells_m)} Elemente')
    else:
        print('Kein Tet4 im MSH → Fallback')
        gmsh_ok = False

if not gmsh_ok:
    points_m, cells_m, ele_type = bbox_hex_mesh(geom_data, MESH_SIZE)
    print(f'\nFallback Hex8-Netz: {len(points_m)} Knoten, {len(cells_m)} Elemente')

cell_type = get_meshio_cell_type(ele_type)
mesh = Mesh(points_m, cells_m)

print(f'\nElement-Typ : {ele_type}')
print(f'Knotenbereich: x=[{points_m[:,0].min():.2f}, {points_m[:,0].max():.2f}] m')
print(f'               y=[{points_m[:,1].min():.2f}, {points_m[:,1].max():.2f}] m')
print(f'               z=[{points_m[:,2].min():.2f}, {points_m[:,2].max():.2f}] m')

  BBox-Netz Balken_B1           :     27 Knoten       8 Elemente
  BBox-Netz Stuetze_C1          :     27 Knoten       8 Elemente
  BBox-Netz Stuetze_C2          :     27 Knoten       8 Elemente

Fallback Hex8-Netz: 81 Knoten, 24 Elemente

Element-Typ : HEX8
Knotenbereich: x=[0.00, 0.01] m
               y=[0.00, 0.00] m
               z=[0.00, 0.00] m


## 3 — Materialparameter & Problem-Klasse

In [ ]:
E  = E_DEFAULT
nu = NU_DEFAULT
mu    = E / (2.0 * (1.0 + nu))
lmbda = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
q     = Q_LOAD

# Ober- und Unterkante des Gesamtmodells
z_min = float(points_m[:, 2].min())
z_max = float(points_m[:, 2].max())
x_min = float(points_m[:, 0].min())
x_max = float(points_m[:, 0].max())

print(f'E={E:.0f} kN/m²  nu={nu}  q={q} kN/m²')
print(f'mu={mu:.1f}  lambda={lmbda:.1f}')
print(f'z: [{z_min:.3f}, {z_max:.3f}] m')


class GesamtstatikProblem(Problem):
    """Lineare Elastizität 3D — GESAMTSTATIK."""

    def get_tensor_map(self):
        def stress(u_grad):
            eps = 0.5 * (u_grad + u_grad.T)
            return lmbda * jnp.trace(eps) * jnp.eye(self.dim) + 2.0 * mu * eps
        return stress

    def get_surface_maps(self):
        def load_top(u, x):
            return jnp.array([0.0, 0.0, q])
        return [load_top]

E=31000 kN/m²  nu=0.2  q=10.0 kN/m²
mu=12916.7  lambda=8611.1
z: [0.000, 0.004] m


## 4 — Randbedingungen

| Auflager | Position | Typ | Gesperrt |
|----------|----------|-----|----------|
| Einspannung (Fundament) | z = z_min | Eingespannt | u_x, u_y, u_z |
| Oberkante | z = z_max | Neumann | Flächenlast q |

> Randbedingungen können in Zelle 5 angepasst werden.

In [ ]:
# Toleranz: halbe kleinste Elementkante
all_lengths = []
for c in cells_m[:min(500, len(cells_m))]:
    pts = points_m[c]
    for i in range(len(pts)):
        for j in range(i+1, len(pts)):
            all_lengths.append(onp.linalg.norm(pts[i] - pts[j]))
tol = float(onp.percentile(all_lengths, 5)) * 0.6 if all_lengths else 0.02
print(f'Toleranz für RB-Erkennung: tol = {tol:.4f} m')

def zero(p): return 0.0

# Einspannung: gesamte Unterkante z ≈ z_min
def at_base(p):
    return jnp.isclose(p[2], z_min, atol=tol)

# Oberkante für Neumann-Last
def at_top(p):
    return jnp.isclose(p[2], z_max, atol=tol)

dirichlet_bc_info = [
    [at_base, at_base, at_base],
    [0,       1,       2      ],
    [zero,    zero,    zero   ],
]

location_fns = [at_top]

n_base = int(onp.sum(onp.abs(points_m[:, 2] - z_min) < tol))
n_top  = int(onp.sum(onp.abs(points_m[:, 2] - z_max) < tol))
print(f'Einspannung: {n_base} Knoten an z={z_min:.3f} m')
print(f'Last-Fläche: {n_top}  Knoten an z={z_max:.3f} m')

## 5 — Lösen

In [ ]:
problem = GesamtstatikProblem(
    mesh=mesh,
    vec=3,
    dim=3,
    ele_type=ele_type,
    dirichlet_bc_info=dirichlet_bc_info,
    location_fns=location_fns,
)

print('Löse Gleichungssystem …')
sol = solver(problem, solver_options={'umfpack_solver': {}})
print('Fertig. Shape:', sol[0].shape)

NameError: name 'dirichlet_bc_info' is not defined

## 6 — Ergebnisse

In [ ]:
ux = onp.array(sol[0][:, 0])
uy = onp.array(sol[0][:, 1])
uz = onp.array(sol[0][:, 2])
u_abs = onp.sqrt(ux**2 + uy**2 + uz**2)

print('=== Verformungsergebnisse ===')
print(f'  w_max (uz min) = {uz.min()*1000:.3f} mm  bei Knoten {uz.argmin()}')
print(f'  w_max (uz max) = {uz.max()*1000:.3f} mm')
print(f'  u_abs_max      = {u_abs.max()*1000:.3f} mm  bei Knoten {u_abs.argmax()}')
print()

# ── Koordinaten des max. Verformungsknotens ────────────────────────────────
k_max = int(u_abs.argmax())
print(f'  Knoten {k_max}: x={points_m[k_max,0]:.3f} m  y={points_m[k_max,1]:.3f} m  z={points_m[k_max,2]:.3f} m')

In [ ]:
# ── Visualisierung: 4 Subplots ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GESAMTSTATIK — Verformungen & Spannung (XZ-Schnitt bei y ≈ Ly/2)', fontsize=13)

y_mid = (points_m[:, 1].min() + points_m[:, 1].max()) / 2
y_tol = (points_m[:, 1].max() - points_m[:, 1].min()) / 8
mask = onp.abs(points_m[:, 1] - y_mid) < y_tol

px = points_m[mask, 0]
pz = points_m[mask, 2]

data_list = [
    (ux[mask] * 1000, 'u_x [mm]', 'RdBu_r'),
    (uy[mask] * 1000, 'u_y [mm]', 'RdBu_r'),
    (uz[mask] * 1000, 'u_z [mm]', 'RdBu'),
    (u_abs[mask] * 1000, '|u| [mm]', 'plasma'),
]

for ax, (vals, label, cmap) in zip(axes.flat, data_list):
    sc = ax.scatter(px, pz, c=vals, cmap=cmap, s=4, rasterized=True)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('z [m]')
    ax.set_title(label)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'vtk' / 'gesamtstatik_verformung.png'), dpi=150)
plt.show()
print('Plot gespeichert:', DATA_DIR / 'vtk' / 'gesamtstatik_verformung.png')

## 7 — Spannungsauswertung (Gauss-Punkte → Knoten)

σ wird an Gauss-Punkten berechnet und via Nearest-Neighbor auf Knoten extrapoliert.

In [ ]:
from scipy.spatial import cKDTree

def compute_stress_nodes(pts, cells, u_sol, lmbda, mu, ele_type):
    """Berechnet Spannungen an Knoten via Gauss-Punkt-Extrapolation."""
    sigma_gp = []   # (n_gp, 6) — Voigt: s11,s22,s33,s12,s13,s23
    gp_coords = []

    if ele_type == 'TET4':
        # Einzelner Gauss-Punkt im Mittelpunkt jedes Tet4
        for cell in cells:
            p = pts[cell]                           # (4,3)
            center = p.mean(axis=0)
            # Verformungen an den 4 Knoten
            ue = u_sol[cell]                        # (4,3)
            # Lineares Tet4: konstante Dehnungen aus Jacobi-Ableitung
            J = p[1:4] - p[0]                      # (3,3)
            try:
                J_inv = onp.linalg.inv(J.T)
            except onp.linalg.LinAlgError:
                continue
            dN = onp.array([[-1,-1,-1],[1,0,0],[0,1,0],[0,0,1]], dtype=float)
            dNdx = dN @ J_inv                       # (4,3)
            grad_u = ue.T @ dNdx                    # (3,3)
            eps = 0.5 * (grad_u + grad_u.T)
            tr_eps = onp.trace(eps)
            sig = lmbda * tr_eps * onp.eye(3) + 2*mu * eps
            sigma_gp.append([sig[0,0],sig[1,1],sig[2,2],sig[0,1],sig[0,2],sig[1,2]])
            gp_coords.append(center)
    else:
        # HEX8: Mittelpunkt jedes Elements
        for cell in cells:
            p = pts[cell]
            center = p.mean(axis=0)
            sigma_gp.append([0.0]*6)
            gp_coords.append(center)

    if not sigma_gp:
        return onp.zeros((len(pts), 6))

    sigma_gp   = onp.array(sigma_gp)
    gp_coords  = onp.array(gp_coords)

    tree = cKDTree(gp_coords)
    _, idx = tree.query(pts)
    return sigma_gp[idx]   # (n_nodes, 6)


print('Spannungsberechnung …')
u_np = onp.array(sol[0])   # (n_nodes, 3)
sigma_nodes = compute_stress_nodes(points_m, cells_m, u_np, float(lmbda), float(mu), ele_type)

# Von-Mises
s = sigma_nodes
s11,s22,s33,s12,s13,s23 = s[:,0],s[:,1],s[:,2],s[:,3],s[:,4],s[:,5]
vm = onp.sqrt(0.5 * ((s11-s22)**2 + (s22-s33)**2 + (s33-s11)**2
                      + 6*(s12**2 + s13**2 + s23**2)))

print(f'σ_xx: min={s11.min():.1f}  max={s11.max():.1f} kN/m²')
print(f'σ_zz: min={s33.min():.1f}  max={s33.max():.1f} kN/m²')
print(f'σ_vM: max={vm.max():.1f} kN/m²  (= {vm.max()/1000:.2f} MN/m²)')

In [ ]:
# ── Spannungsplot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('GESAMTSTATIK — Spannungen (XZ-Schnitt)', fontsize=12)

data_stress = [
    (s11[mask], 'σ_xx [kN/m²]', 'RdBu_r'),
    (s33[mask], 'σ_zz [kN/m²]', 'RdBu_r'),
    (vm[mask],  'σ_vM [kN/m²]', 'plasma'),
]

for ax, (vals, label, cmap) in zip(axes, data_stress):
    sc = ax.scatter(px, pz, c=vals, cmap=cmap, s=4, rasterized=True)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('z [m]')
    ax.set_title(label)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'vtk' / 'gesamtstatik_spannung.png'), dpi=150)
plt.show()

## 8 — Export (VTK + JSON)

Ergebnisse werden als:
- `gesamtstatik.vtu` → ParaView / PyVista
- `gesamtstatik_results.json` → geobim.app / CesiumJS (Farbgebung nach σ_vM)

In [ ]:
from jax_fem.utils import save_sol

# VTK-Export
vtu_path = str(DATA_DIR / 'vtk' / 'gesamtstatik.vtu')
save_sol(problem.fes[0], sol[0], vtu_path)
print('VTK gespeichert:', vtu_path)

# JSON-Export für CesiumJS / geobim.app
results_json = {
    'meta': {
        'source'    : ifc_path_actual,
        'ele_type'  : ele_type,
        'n_nodes'   : int(len(points_m)),
        'n_elements': int(len(cells_m)),
        'E_kNm2'    : E,
        'nu'        : nu,
        'q_kNm2'    : q,
        'units'     : 'kN + m',
    },
    'summary': {
        'uz_max_mm'    : float(round(uz.min() * 1000, 3)),
        'u_abs_max_mm' : float(round(u_abs.max() * 1000, 3)),
        'sigma_vm_max' : float(round(vm.max(), 1)),
        'sigma_xx_min' : float(round(s11.min(), 1)),
        'sigma_xx_max' : float(round(s11.max(), 1)),
    },
    'per_bauteil': []
}

for gd in geom_data:
    bmin, bmax = gd['bbox']
    # Knoten in BBox dieses Bauteils
    mask_b = (
        (points_m[:, 0] >= bmin[0] - tol) & (points_m[:, 0] <= bmax[0] + tol) &
        (points_m[:, 1] >= bmin[1] - tol) & (points_m[:, 1] <= bmax[1] + tol) &
        (points_m[:, 2] >= bmin[2] - tol) & (points_m[:, 2] <= bmax[2] + tol)
    )
    if mask_b.sum() == 0:
        continue
    results_json['per_bauteil'].append({
        'name'         : gd['name'],
        'type'         : gd['type'],
        'guid'         : gd['guid'],
        'uz_max_mm'    : float(round(uz[mask_b].min() * 1000, 3)),
        'u_abs_max_mm' : float(round(u_abs[mask_b].max() * 1000, 3)),
        'sigma_vm_max' : float(round(vm[mask_b].max(), 1)),
    })

json_path = DATA_DIR / 'gesamtstatik_results.json'
with open(json_path, 'w') as f:
    json.dump(results_json, f, indent=2)
print('JSON gespeichert:', json_path)
print()
print('=== Zusammenfassung ===')
for bauteil in results_json['per_bauteil']:
    print(f"  {bauteil['type']:12s} {bauteil['name']:20s}  "
          f"w={bauteil['uz_max_mm']:7.3f} mm  σ_vM={bauteil['sigma_vm_max']:.1f} kN/m²")

## 9 — Nächste Schritte

| Schritt | Beschreibung |
|---------|-------------|
| **A** | Eigene IFC einbinden: `IFC_PATH` in Zelle 0 setzen, `USE_DEMO_MODEL = False` |
| **B** | IFC vom Server herunterladen: `server/ifc_to_fem.py` aufrufen (wird noch erstellt) |
| **C** | Material per IFC-Property: `IfcMaterialLayerSet` → E-Modul-Mapping |
| **D** | Randbedingungen aus IFC: `IfcRelConnectsStructuralMember` auslesen |
| **E** | Ergebnisse in CesiumJS einfärben: JSON `per_bauteil.sigma_vm_max` → Property-Table |
| **F** | PyVista-3D-Visualisierung des VTK-Ergebnisses hinzufügen |

```python
# Schnellstart Download vom geobim.app-Server:
import urllib.request
urllib.request.urlretrieve(
    'https://geobim.app/api/ifc/ASSET_ID',
    '../data/gesamtstatik.ifc'
)
```